In [ ]:
# Phase 1 - Imports
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
REVIEWS_DATA = os.getenv("REVIEWS_DATA")

import re
import random
import numpy as np
import pandas as pd
import pickle
from statistics import mode
import nltk
from nltk import word_tokenize
from nltk.stem import LancasterStemmer
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from bs4 import BeautifulSoup

In [ ]:
# Phase 2 - Loading and Preprocessing the Data (This cleans the data and prepares it for training the model.)
# Load the dataset
data = pd.read_csv(REVIEWS_DATA, encoding='utf-8')
# Drop duplicate rows with 'Text' column
data.drop_duplicates(subset=['Text'], inplace=True)
# Drop any rows with missing values
data.dropna(inplace=True)
data = data.head(100)
# Extract the 'Text' columns as input data
texts = data['Text'].tolist() # Converts the 'Text' column to a list of strings
# Extract the 'Summary' column
summary = data['Summary'].tolist() # Converts the 'Summary' column to a list of strings
# Replace any empty string summaries with NaN, then drop those rows too
data['Summary'] = data['Summary'].replace('', np.nan)
data.dropna(subset=['Summary'], inplace=True)

In [ ]:
# Phase 3 - Preprocessing the Text Data (This includes tokenization, stemming, and removing stop words.)

# Step 1: Setting up tools:
file_path = REVIEWS_DATA
try:
    with open(file_path, 'r', encoding='utf-8') as file:
        reviews_data = pd.read_csv(file)
        print("Data loaded successfully.")
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred while loading the data: {e}")
# intialize lancaster stemmer and stop words
stemmer = LancasterStemmer()
stop_words = set(stopwords.words('english'))
# Intialize NLKT's WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [ ]:
# Phase 2: Step 2
"""
Step 2: A clean function that labeles inputs or outputs as either text or summary and then cleans them accordingly. This includes removing HTML tags, special characters, and stop words, as well as performing stemming and lemmatization.
@param text: The input text to be cleaned.
@return: A list of cleaned tokens.
"""
def clean_text(text):
    #Remove HTML tags using B4S
    soup = BeautifulSoup(text, "html.parser")
    cleaned_text = soup.get_text()
    # Remove special characters and numbers using regex
    cleaned_text = re.sub(r'[^a-zA-Z\s]', '', cleaned_text)
    # tokenize the cleaned text
    tokens = [word.lower() for word in word_tokenize(cleaned_text)]
    # Filter out words that are shorter than 3 chars
    tokens = [word for word in tokens if len(word) > 2]
    # Expand contractiosn using dictionary
    contractions = {
        "can't": "cannot",
        "won't": "will not",
        "n't": " not",
        "'re": " are",
        "'s": " is",
        "'d": " would",
        "'ll": " will",
        "'t": " not",
        "'ve": " have",
        "'m": " am"
    }
    # check if sours is inputs: if it is remove stopwords
    if text in texts:
        tokens = [contractions.get(word, word) for word in tokens if word not in stop_words]
    # Check if source is a target: if it si only remove stop words
    elif text in summary:
        tokens = [contractions.get(word, word) for word in tokens if word not in stop_words]
    # finally return the list of cleaned tokens
    return tokens

In [ ]:
# Phase 2: Step 3

# Step 3: Loop through the tokens and apply stemming and lemmatization to each word. This will help reduce the vocabulary size and improve the model's ability to generalize.
for text in texts:
    cleaned_tokens = clean_text(text)
    # Apply stemming and lemmatization
    stemmed_tokens = [stemmer.stem(word) for word in cleaned_tokens]
    # Apply lemmatization to the cleaned tokens
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in cleaned_tokens]


In [ ]:
# Phase 2: Step 4: Deduplication and Final Cleaning
# Step 4: After stemming and lemmatization, we will have a list of tokens for each text. We can then deduplicate these tokens to further reduce the vocabulary size. Finally, we can join the cleaned tokens back into a single string for each text, which will be used as input to the model.
cleaned_texts = []
for text in texts:
    cleaned_tokens = clean_text(text)
    cleaned_texts.append(' '.join(cleaned_tokens))

    # Count unque tokens
    unique_tokens = set(cleaned_tokens)
    # Find the most common token
    most_common_token = mode(cleaned_tokens)
    # Find input and target length of tokens
    input_length = len(cleaned_tokens)
    target_length = len(clean_text(summary[texts.index(text)]))
    # Printing all 4 vals
    print(f"Unique tokens: {len(unique_tokens)}, Most common token: '{most_common_token}', Input length: {input_length}, Target length: {target_length}")
    

In [ ]:
# Phase 4: words -> numbers for the model to process them 
# Step 1: Training out data and splitting it

# Split our input texts
def split_data(texts, summary, test_size=0.2, random_state=0):
    return train_test_split(texts, summary, test_size=test_size, random_state=random_state)

# Step 2: mapping words to integers using our dict we made

# Mapes each word to unique integer based on its index in the set of all unique words
words_to_int = {word: idx for idx, word in enumerate(set(word for text in cleaned_texts for word in text.split()))}
# Reverse our mapping to get int to word
int_texts = [[words_to_int[word] for word in text.split()] for text in cleaned_texts]

"""
# Step 3: our text -> seq function
@param texts: A list of cleaned text strings to be converted into sequences of integers.
@param words_to_int: A dictionary mapping each unique word to a unique integer index.
@return: A list of lists, where each inner list contains the integer indices corresponding to the words in the input text.
"""
def texts_to_sequences(texts, words_to_int) -> list:
    return [[words_to_int.get(word, 0) for word in text.split()] for text in texts]

# Step 4: pad seq function this function is used to pad our sequences to a fixed length so that they can be processed by the model.
def pad_sequences(seqs, max_len):
    return nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)

# Step 5: Preparing the encoder and decoder data
x_train, x_test, y_train, y_test = train_test_split(int_texts, summary, test_size=0.2, random_state=0)

# Our input set (already integers)
x_train_seq = x_train
x_test_seq = x_test

# our target set (still strings, needs converting)
y_train_seq = texts_to_sequences(y_train, words_to_int)
y_test_seq = texts_to_sequences(y_test, words_to_int)

# Pad all sequences into tensors
x_train_padded = nn.utils.rnn.pad_sequence([torch.tensor(seq) for seq in x_train_seq], batch_first=True, padding_value=0)
x_test_padded = nn.utils.rnn.pad_sequence([torch.tensor(seq) for seq in x_test_seq], batch_first=True, padding_value=0)
y_train_padded = nn.utils.rnn.pad_sequence([torch.tensor(seq) for seq in y_train_seq], batch_first=True, padding_value=0)
y_test_padded = nn.utils.rnn.pad_sequence([torch.tensor(seq) for seq in y_test_seq], batch_first=True, padding_value=0)

# Step 6: Pytorch Dataset and DataLoader
class TextSummaryDataset(Dataset):
    def __init__(self, x_data, y_data):
        self.x_data = x_data
        self.y_data = y_data

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]

# Step 7: Creating the dataloader ( batch size = 612, shuffle is true)
train_set = TextSummaryDataset(x_train_padded, y_train_padded)
test_set = TextSummaryDataset(x_test_padded, y_test_padded)
train_loader = DataLoader(train_set, batch_size=612, shuffle=True)
test_loader = DataLoader(test_set, batch_size=612, shuffle=False)

print("Data loading and preprocessing complete.")
batch_x, batch_y = next(iter(train_loader))
assert batch_x.shape[0] == batch_y.shape[0], "Batch sizes don't match!"
print(f"Input tensor shape: {batch_x.shape}")
print(f"Target tensor shape: {batch_y.shape}")
print("Total Number of batches:", len(train_loader))


In [ ]:
# Phase 5: Building the Seq2Seq Model with Attention

latent_dim = 500
num_in_words = len(words_to_int)
num_tr_words = len(words_to_int)

#  ENCODER 
# __init__ layers to build:
#   - nn.Embedding(num_in_words + 1, latent_dim, padding_idx=0)
#   - Three separate nn.LSTM(latent_dim, latent_dim, batch_first=True)
#     named lstm1, lstm2, lstm3
#
# forward(self, x):
#   - Embed x
#   - Pass through lstm1, then lstm2, then lstm3 (chain the outputs manually)
#   - Return: out3 (all hidden states from lstm3), h3, c3 (final hidden & cell state)

class Encoder(nn.Module):
    def __init__(self, num_in_words, latent_dim):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(num_in_words + 1, latent_dim, padding_idx=0)
        self.lstm1 = nn.LSTM(latent_dim, latent_dim, batch_first=True)
        self.lstm2 = nn.LSTM(latent_dim, latent_dim, batch_first=True)
        self.lstm3 = nn.LSTM(latent_dim, latent_dim, batch_first=True)
        

    def forward(self, x):
        embed = self.embedding(x)
        out1, (h1, c1) = self.lstm1(embed)
        out2, (h2, c2) = self.lstm2(out1)
        out3, (h3, c3) = self.lstm3(out2)
        return out3, h3, c3


# ─── ATTENTION (Luong dot-product) ────────────────────────────────────────────
# No __init__ needed (no learnable parameters)
#
# forward(self, decoder_out, encoder_out):
#   - Compute similarity scores using torch.bmm
#       decoder_out: [batch, dec_len, latent_dim]
#       encoder_out: [batch, enc_len, latent_dim]
#       scores shape should be: [batch, dec_len, enc_len]
#       Hint: you need to transpose encoder_out before bmm
#   - Apply softmax over the enc_len dimension to get attention weights
#   - Multiply weights by encoder_out to get the context vector
#       context shape: [batch, dec_len, latent_dim]
#   - Return context

class Attention(nn.Module):
    def forward(self, decoder_out, encoder_out):
        decoder_out = decoder_out.contiguous()
        encoder_out = encoder_out.contiguous()
        scores = torch.bmm(decoder_out, encoder_out.transpose(1, 2))
        attn_weights = torch.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_out)
        return context


# ─── DECODER ──────────────────────────────────────────────────────────────────
# __init__ layers to build:
#   - nn.Embedding(num_tr_words + 1, latent_dim, padding_idx=0)
#   - nn.LSTM(latent_dim, latent_dim, batch_first=True)
#   - An instance of your Attention class
#   - nn.Linear(latent_dim * 2, num_tr_words + 1)
#     (*2 because you will concatenate decoder output + context before the linear layer)
#
# forward(self, x, h, c, encoder_out):
#   - Embed x
#   - Run through lstm, passing h and c as the initial state
#   - Compute attention context using encoder_out and the decoder's lstm output
#   - Concatenate lstm output and context along the last dimension
#   - Pass through the linear layer to get word probability logits
#   - Return: output logits, new h, new c

class Decoder(nn.Module):
    def __init__(self, num_tr_words, latent_dim):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(num_tr_words + 1, latent_dim, padding_idx=0)
        self.lstm = nn.LSTM(latent_dim, latent_dim, batch_first=True)
        self.attention = Attention()
        self.fc = nn.Linear(latent_dim * 2, num_tr_words + 1)

    def forward(self, x, h, c, encoder_out):
        embed = self.embedding(x)
        lstm_out, (h, c) = self.lstm(embed, (h, c))
        context = self.attention(lstm_out, encoder_out)
        combined = torch.cat([lstm_out, context], dim=-1)
        logits = self.fc(combined)
        return logits, h, c


# ─── INSTANTIATE ──────────────────────────────────────────────────────────────
# - Detect device (GPU if available, else CPU) using torch.device
# - Create encoder and decoder instances, move both to device with .to(DEVICE)
# - Print total trainable parameter count across both models

DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f'Using device: {DEVICE}')